# Evaluación de PC-SMOTE con Grid Search en el dataset Shuttle (Generación de caso base y datasets aumentados)


In [16]:
import sys
sys.path.append("../scripts")
sys.path.append("../datasets")

import os

# Rutas de datasets y resultados
RUTA_DATASETS_BASE = "../datasets/datasets_aumentados/base/"
RUTA_DATASETS_AUMENTADOS = "../datasets/datasets_aumentados/"
RUTA_DATASETS_CLASICOS = "../datasets/datasets_aumentados/resampler_clasicos/"
DIRECTORIO_SALIDA = "../resultados"

os.makedirs(DIRECTORIO_SALIDA, exist_ok=True)
os.makedirs(RUTA_DATASETS_CLASICOS, exist_ok=True)


In [17]:
import gc, time  # gc: liberación explícita de memoria entre ejecuciones; time: medición de duración de búsquedas
from dataclasses import dataclass, asdict  # dataclass: estructura limpia para registrar resultados y metadatos de cada combinación
import json  # guardar resultados intermedios en formato JSON

import numpy as np  # operaciones numéricas y manipulación de vectores/matrices
import pandas as pd  # manejo de estructuras tabulares (dataframes) para consolidar resultados


# Utilizamos validación estratificada + búsqueda aleatoria de hiperparámetros
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV

# Métricas utilizadas en CV y test (todas macro para evitar sesgos por clase mayoritaria)
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    recall_score,
    accuracy_score,
    make_scorer
)

# Cada modelo se ejecuta dentro de un Pipeline para permitir transformaciones futuras
from sklearn.pipeline import Pipeline

# Modelo principal evaluado (Random Forest)
from sklearn.ensemble import RandomForestClassifier

# Suprimir warnings de convergencia innecesarios (SVM no se usa en esta fase)
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Controlar comportamiento en entornos con múltiples núcleos
# (evita paralelismo interno conflictivo con n_jobs de sklearn)
import os

# Estado aleatorio fijo para reproducibilidad entre ejecuciones
RANDOM_STATE = 42
       
# En Shuttle aumentado omitimos SVM por inestabilidad del ROC-AUC y límites computacionales
OMITIR_SVM_EN_SHUTTLE_AUMENTADO = True

# Archivo Excel consolidado con resultados CV y Test para todas las técnicas
NOMBRE_ARCHIVO_EXCEL = os.path.join(DIRECTORIO_SALIDA, "resultados_RS_cv_vs_test.xlsx")

In [18]:
from pathlib import Path
import re

# =========================
# Estructuras de datos
# =========================
@dataclass
class DatasetCombination:
    dataset_logico: str
    tipo_combination: str      # "base" | "clasico" | "pcsmote"
    ruta_train_csv: str
    ruta_test_csv: str

    tecnica_aumento: str = "base"

    # Parámetros genéricos (clásicos / base)
    valor_densidad: str | int | None = "--"
    valor_riesgo: str | int | None = "--"
    criterio_pureza: str | None = "--"

    # Parámetros PCSMOTE (si aplica)
    percentil_radio_distancia: str | int | None = "--"
    percentil_riesgo: str | int | None = "--"
    umbral_densidad: str | None = "--"
    umbral_riesgo: str | None = "--"

    tipo_pureza: str = "--"           # PE.. o Upp.. del nombre de archivo
    nombre_configuracion: str = ""    # PRD.._PR.._CP.._UD.._UR.._.._I.._SV.._SG..

    grado_limpieza: str | int = "--"  # I0, I1, I3, etc.
    total_muestras_train: int | None = None
    tamanio_dataset: int | None = None  # tamaño total train+test (si lo tenés)
    sinteticos_generados: int = 0
    semillas_validas: int = 0
    semillas_analizadas: int = 0


@dataclass
class RegistroRendimiento:
    dataset_logico: str
    tipo_combination: str
    nombre_modelo_aprendizaje: str
    nombre_configuracion: str
    tecnica_aumento: str
    valor_densidad: str
    valor_riesgo: str
    criterio_pureza: str
    grado_limpieza: str

    cantidad_train: int
    cantidad_test: int
    cantidad_caracteristicas: int

    # Métricas CV
    cv_f1_macro: float
    cv_f1_weighted: float
    cv_balanced_accuracy: float
    cv_recall_macro: float
    cv_accuracy: float

    # Métricas Test
    test_f1_macro: float
    test_f1_weighted: float
    test_balanced_accuracy: float
    test_recall_macro: float
    test_accuracy: float
    
    mejores_hiperparametros: str
    tiempo_busqueda_seg: float

    # semillas_sinteticas_candidatas: int 
    # semillas_candidatas_validas_train: int



def enumerar_combinaciones_base_y_aumentadas(
    ruta_base,
    ruta_clasicos,
    ruta_aumentados,
    verbose=True
):
    combinaciones = []
    cont_combinaciones = 0

    # Mapear (dataset_logico, grado_limpieza) → tamaño_train_base
    tamanio_train_base_por_dataset_y_I = {}

    # ==========================================================
    # 1) BASE
    #    train: {dataset}_I{I}_tm{n}_train.csv
    #    test : {dataset}_tm{n}_test.csv
    # ==========================================================
    if verbose:
        print(f"📂 Explorando carpeta base: {ruta_base}")

    archivos_base = os.listdir(ruta_base)

    for nombre in archivos_base:
        if not nombre.endswith("_train.csv"):
            if verbose:
                print(f"  ⚪ Omitido (no es *_train.csv): {nombre}")
            continue

        m = re.match(r"(.+?)_I(\d+)_tm(\d+)_train\.csv$", nombre)
        if not m:
            if verbose:
                print(f"  ⚪ No coincide patrón base con I*_tm*_train: {nombre}")
            continue

        dataset_logico = m.group(1)
        grado_limpieza = int(m.group(2))
        total_muestras_train = int(m.group(3))

        # Registrar tamaño de train base para este (dataset, I)
        clave_base = (dataset_logico, grado_limpieza)
        tamanio_train_base_por_dataset_y_I[clave_base] = total_muestras_train

        ruta_train_csv = os.path.join(ruta_base, nombre)

        # Buscar test correspondiente: {dataset}_tm{n}_test.csv
        patron_test = re.compile(rf"^{re.escape(dataset_logico)}_tdataset(\d+)_tm(\d+)_test\.csv$")
        nombre_test = None
        n_test_detectado = None

        for nombre_candidato in archivos_base:
            m_test = patron_test.match(nombre_candidato)
            if m_test:
                # Si hubiera más de uno, nos quedamos con el de mayor tm
                n_tm = int(m_test.group(2))
                tamanio_dataset=int(m_test.group(1))

                if n_test_detectado is None or n_tm > n_test_detectado:
                    n_test_detectado = n_tm
                    nombre_test = nombre_candidato

        if nombre_test is None:
            if verbose:
                print(f"  ⚠️  Falta test para dataset base '{dataset_logico}', se omite {nombre}")
            continue

        ruta_test_csv = os.path.join(ruta_base, nombre_test)

        cont_combinaciones += 1
        print(f"#{cont_combinaciones}  ✅ Agregado base: {nombre} combinado con {nombre_test}")

        combinaciones.append(DatasetCombination(
            dataset_logico=dataset_logico,
            tipo_combination="base",
            ruta_train_csv=ruta_train_csv,
            ruta_test_csv=ruta_test_csv,
            tecnica_aumento="base",
            valor_densidad=None,
            valor_riesgo=None,
            criterio_pureza=None,
            grado_limpieza=grado_limpieza,
            total_muestras_train=total_muestras_train,
            tamanio_dataset=tamanio_dataset
        ))

    # ==========================================================
    # 2) CLÁSICOS
    #    {tecnica}_{dataset}_I{I}_sg{sg}_train.csv
    #    test base: {dataset}_tm{n}_test.csv (mismo criterio que base)
    # ==========================================================
    if verbose:
        print(f"📂 Explorando carpeta clásicos: {ruta_clasicos}")

    archivos_clasicos = os.listdir(ruta_clasicos)

    for nombre in archivos_clasicos:
        if not nombre.endswith("_train.csv"):
            continue

        # ejemplo: adasyn_us_crime_I1_sg120_train.csv
        m = re.match(r"(.+?)_(.+?)_I(\d+)_sg(\d+)_train\.csv$", nombre)
        if not m:
            if verbose:
                print(f"  ⚠️  No cumple patrón clásicos: {nombre}")
            continue

        tecnica = m.group(1)
        dataset_logico = m.group(2)
        grado_limpieza = int(m.group(3))
        sinteticos_generados = int(m.group(4))

        # Recuperar tamaño de train base para este dataset y este I
        clave_base = (dataset_logico, grado_limpieza)
        total_muestras_train = tamanio_train_base_por_dataset_y_I.get(clave_base)

        if total_muestras_train is None:
            if verbose:
                print(
                    f"  ⚠️  No se encontró tamaño de train base para "
                    f"(dataset='{dataset_logico}', I={grado_limpieza}). Se omite {nombre}"
                )
            continue
        
        ruta_train_csv = os.path.join(ruta_clasicos, nombre)

        # Buscar test correspondiente en carpeta base
        patron_test = re.compile(rf"^{re.escape(dataset_logico)}_tdataset(\d+)_tm(\d+)_test\.csv$")
        nombre_test = None
        n_test_detectado = None

        for nombre_candidato in archivos_base:
            m_test = patron_test.match(nombre_candidato)
            if m_test:
                n_tm = int(m_test.group(2))
                tamanio_dataset = int(m_test.group(1))
                if n_test_detectado is None or n_tm > n_test_detectado:
                    n_test_detectado = n_tm
                    nombre_test = nombre_candidato

        if nombre_test is None:
            if verbose:
                print(f"  ⚠️  No hay test base para dataset '{dataset_logico}', se omite {nombre}")
            continue

        ruta_test_csv = os.path.join(ruta_base, nombre_test)

        cont_combinaciones += 1
        print(f"#{cont_combinaciones}  ✅ Agregado clásico: {nombre} combinado con {nombre_test}")

        combinaciones.append(DatasetCombination(
            dataset_logico=dataset_logico,
            tipo_combination="clasico",
            ruta_train_csv=ruta_train_csv,
            ruta_test_csv=ruta_test_csv,
            tecnica_aumento=tecnica.lower(),
            valor_densidad=None,
            valor_riesgo=None,
            criterio_pureza=None,
            grado_limpieza=grado_limpieza,
            total_muestras_train=total_muestras_train,
            sinteticos_generados=sinteticos_generados,
            tamanio_dataset=tamanio_dataset
        ))

    # ==========================================================
    # 3) PC-SMOTE (nuevo patrón)
    #
    # pcs_{dataset}_PRD{prd}_PR{pr}_CP{ent|prop}_UD{ud3}_{PE..|Ppp..}_I{iso}_SG{sg}_train.csv
    #
    # Ej:
    #   pcs_ecoli_PRD35_PR35_CPent_UD080_PE45_I0_SG120_train.csv
    #   pcs_ecoli_PRD35_PR35_CPprop_UD080_Ppp041_I0_SG007_train.csv
    #
    # valor_densidad  → percentil radio distancia (PRD)
    # valor_riesgo    → percentil riesgo (PR)
    # criterio_pureza → "entropia" / "proporcion"
    # grado_limpieza  → iso (I*)
    # sinteticos_generados → SG
    # ==========================================================
    if verbose:
        print(f"📂 Explorando carpeta aumentados: {ruta_aumentados}")

    archivos_aumentados = os.listdir(ruta_aumentados)

    patron_pcsmote = re.compile(
        r"^pcs_(?P<dataset>.+?)_"
        r"PRD(?P<prd>\d+)_"
        r"PR(?P<pr>\d+)_"
        r"CP(?P<cp>(?:ent|prop))_"
        r"UD(?P<ud>\d{3})_"
        r"(?P<tipo_pureza>(?:PE\d+|Upp\d{3}))_"
        r"UR(?P<ur>\d{3})_"
        r"I(?P<iso>\d+)_"
        r"SC(?P<sc>\d+)_"
        r"SV(?P<sv>\d+)_"
        r"SG(?P<sg>\d+)_train\.csv$"
    )

    for nombre in archivos_aumentados:
        if not nombre.endswith("_train.csv"):
            continue

        m = patron_pcsmote.match(nombre)
        if not m:
            if verbose:
                print(f"  ⚪ Omitido (no es pcs válido): {nombre}")
            continue

        dataset_logico = m.group("dataset")
        valor_densidad = int(m.group("prd"))   # percentil radio distancia
        valor_riesgo   = int(m.group("pr"))    # percentil riesgo
        cp_code        = m.group("cp")         # "ent" | "prop"
        ud_str       = m.group("ud")        # umbral densidad en %, si después lo querés usar
        ur_str       = m.group("ur")        # umbral densidad en %, si después lo querés usar
        tipo_pureza = m.group("tipo_pureza")  # PE.. / Ppp.., si lo necesitás luego
        grado_limpieza = int(m.group("iso"))   # I*
        semillas_analizadas = int(m.group("sc"))
        semillas_validas = int(m.group("sv"))
        sinteticos_generados = int(m.group("sg"))

        print(f"  ➡️  Descifrado pcsmote: dataset={dataset_logico}, prd={valor_densidad}, pr={valor_riesgo}, cp={cp_code}, ud={ud_str}, ur={ur_str}, tipo_pureza={tipo_pureza}, I={grado_limpieza}, sv={semillas_validas}, sg={sinteticos_generados}")

        if cp_code == "ent":
            criterio_pureza = "entropia"
        else:
            criterio_pureza = "proporcion"

        # nombre_configuracion EXACTO según el patrón
        nombre_configuracion = (
            f"PRD{valor_densidad}_"
            f"PR{valor_riesgo}_"
            f"CP{cp_code}_"
            f"UD{ud_str}_"
            f"UR{ur_str}_"
            f"{tipo_pureza}_"
            f"I{grado_limpieza}_"
            f"SC{semillas_analizadas}_"
            f"SV{semillas_validas}_"
            f"SG{sinteticos_generados}"
        )            

        ruta_train_csv = os.path.join(ruta_aumentados, nombre)

        # Buscar test correspondiente en carpeta base
        patron_test = re.compile(rf"^{re.escape(dataset_logico)}_tdataset(\d+)_tm(\d+)_test\.csv$")
        nombre_test = None
        n_test_detectado = None

        for nombre_candidato in archivos_base:
            m_test = patron_test.match(nombre_candidato)
            if m_test:
                n_tm = int(m_test.group(2))
                tamanio_dataset = int(m_test.group(1))
                if n_test_detectado is None or n_tm > n_test_detectado:
                    n_test_detectado = n_tm
                    nombre_test = nombre_candidato

        if nombre_test is None:
            if verbose:
                print(f"  ⚠️  No hay test base para dataset '{dataset_logico}', se omite {nombre}")
            continue

        ruta_test_csv = os.path.join(ruta_base, nombre_test)

        cont_combinaciones += 1
        print(f"#{cont_combinaciones}  ✅ Agregado pcsmote: {nombre} combinado con {nombre_test}")

        combinaciones.append(DatasetCombination(
            dataset_logico=dataset_logico,
            tipo_combination="pcsmote",
            ruta_train_csv=ruta_train_csv,
            ruta_test_csv=ruta_test_csv,
            tecnica_aumento="pcsmote",
            valor_densidad=valor_densidad,
            valor_riesgo=valor_riesgo,

            criterio_pureza=criterio_pureza,
            percentil_radio_distancia=valor_densidad,
            percentil_riesgo=valor_riesgo,  
            umbral_densidad=ud_str,
            umbral_riesgo=ur_str,

            grado_limpieza=grado_limpieza,
            sinteticos_generados=sinteticos_generados,
            semillas_validas=semillas_validas,
            semillas_analizadas=semillas_analizadas,
            tipo_pureza=tipo_pureza,                 
            nombre_configuracion=nombre_configuracion,    
            tamanio_dataset=tamanio_dataset
    
        ))

    if verbose:
        print(f"📊 Total combinaciones descubiertas: {len(combinaciones)}")

    return combinaciones


print("🔎 Enumerando combinaciones base y aumentadas...")

combinaciones = enumerar_combinaciones_base_y_aumentadas(
    ruta_base=RUTA_DATASETS_BASE,
    ruta_clasicos=RUTA_DATASETS_CLASICOS,
    ruta_aumentados=RUTA_DATASETS_AUMENTADOS,
    verbose=True
)

if not combinaciones:
    print("❌ No se encontraron combinaciones de datasets.")


datasets_con_base = {c.dataset_logico for c in combinaciones if c.tipo_combination == "base"}
if not datasets_con_base:
    print("❌ No hay datasets base para comparar.")



🔎 Enumerando combinaciones base y aumentadas...
📂 Explorando carpeta base: ../datasets/datasets_aumentados/base/
  ⚪ Omitido (no es *_train.csv): analisis_test.ipynb
#1  ✅ Agregado base: shuttle_I0_tm46400_train.csv combinado con shuttle_tdataset58000_tm11600_test.csv
#2  ✅ Agregado base: shuttle_I1_tm45932_train.csv combinado con shuttle_tdataset58000_tm11600_test.csv
  ⚪ Omitido (no es *_train.csv): shuttle_tdataset58000_tm11600_test.csv
#3  ✅ Agregado base: telco_churn_I0_tm5634_train.csv combinado con telco_churn_tdataset7043_tm1409_test.csv
#4  ✅ Agregado base: telco_churn_I1_tm5577_train.csv combinado con telco_churn_tdataset7043_tm1409_test.csv
  ⚪ Omitido (no es *_train.csv): telco_churn_tdataset7043_tm1409_test.csv
#5  ✅ Agregado base: us_crime_I0_tm1595_train.csv combinado con us_crime_tdataset1994_tm399_test.csv
#6  ✅ Agregado base: us_crime_I1_tm1578_train.csv combinado con us_crime_tdataset1994_tm399_test.csv
#7  ✅ Agregado base: us_crime_I3_tm1546_train.csv combinado con 

In [19]:
EXCLUIR_DATASETS = {
    # "shuttle",
    "iris",
    "glass",
    "heart",
    "wdbc",
    "ecoli",
    # "us_crime",
    "predict_faults",
    "gear_vibration",
    # "telco_churn",
}

def construir_lista_plana_de_tareas(dataset_combinations, orden_modelos,
                                    excluir_datasets=EXCLUIR_DATASETS, verbose=True):
    """
    Lista plana de tareas (modelo, combinación) aplicando exclusión por dataset.
    Nota: La política SVM/Shuttle no aplica acá porque esta notebook evalúa RandomForest.
    """
    tareas = []
    excluidos_por_dataset = 0

    for nombre_modelo in orden_modelos:
        for combo in dataset_combinations:
            ds = combo.dataset_logico.lower()

            if ds in (excluir_datasets or set()):
                excluidos_por_dataset += 1
                continue

            tareas.append((nombre_modelo, combo))

    if verbose:
        print(f"🧮 Tareas planificadas: {len(tareas)} (excluidos por dataset: {excluidos_por_dataset})")
    return tareas


def construir_estimador_y_espacio_random_forest():
    est = Pipeline([
        ('classifier', RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
            bootstrap=True,
            oob_score=False,
            n_estimators=150,
            max_depth=None,
            max_features='sqrt',
            min_samples_split=2,
            min_samples_leaf=1,
            class_weight=None,
            criterion='gini'
        ))
    ])

    # Espacio chico y controlado (4 combinaciones posibles)
    space = {
        "classifier__n_estimators": [150, 300],
        "classifier__max_features": ["sqrt", "log2"],
    }
    return est, space


REGISTRO_MODELOS = {
    "RandomForest": construir_estimador_y_espacio_random_forest,
}
ORDEN_MODELOS = ["RandomForest"]


tareas = construir_lista_plana_de_tareas(
    dataset_combinations=combinaciones,
    orden_modelos=ORDEN_MODELOS,
    excluir_datasets=EXCLUIR_DATASETS,
    verbose=True
)

total_tareas = len(tareas)
print(f"📦 Total de tareas planificadas: {total_tareas}")


🧮 Tareas planificadas: 50 (excluidos por dataset: 0)
📦 Total de tareas planificadas: 50


In [20]:
# Scoring para RandomizedSearchCV
SCORING_REFIT = "f1_macro"
SCORING_MULTIPLE = {
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy",
    "recall_macro": make_scorer(recall_score, average="macro"),
    "accuracy": "accuracy",
}


def _contar_combinaciones_posibles_en_space(space):
    """
    Cuenta combinaciones si space es un dict de listas discretas.
    Si encuentra algo que no es lista/tupla/np.ndarray, devuelve None.
    """
    total = 1
    for k, v in space.items():
        if isinstance(v, (list, tuple, np.ndarray)):
            total *= len(v)
        else:
            return None
    return total


def ejecutar_rs_y_comparar_cv_con_test(
    estimator,
    space,
    X_train,
    y_train,
    X_test,
    y_test,
    configuracion_busqueda,
    verbose=0,
):
    """
    Ejecuta RandomizedSearchCV y devuelve:
      - mejores params
      - tiempo
      - métricas CV del mejor candidato
      - métricas Test con best_estimator_
    """
    inicio = time.perf_counter()

    n_iter_solicitado = int(configuracion_busqueda["n_iter"])
    n_posibles = _contar_combinaciones_posibles_en_space(space)

    # Si el space es discreto (como tu RF), no tiene sentido pedir más iteraciones que combinaciones
    if n_posibles is not None:
        n_iter_efectivo = min(n_iter_solicitado, n_posibles)
    else:
        n_iter_efectivo = n_iter_solicitado

    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=space,
        n_iter=n_iter_efectivo,
        scoring=SCORING_MULTIPLE,
        refit=SCORING_REFIT,
        cv=configuracion_busqueda["cv"],
        random_state=RANDOM_STATE,
        n_jobs=configuracion_busqueda["n_jobs"],
        verbose=verbose,
    )

    search.fit(X_train, y_train)
    elapsed = time.perf_counter() - inicio

    cv_results = search.cv_results_
    best_idx = search.best_index_

    cv_f1       = float(cv_results["mean_test_f1_macro"][best_idx])
    cv_f1_w     = float(cv_results["mean_test_f1_weighted"][best_idx])  
    cv_bacc     = float(cv_results["mean_test_balanced_accuracy"][best_idx])
    cv_recall_m = float(cv_results["mean_test_recall_macro"][best_idx])
    cv_acc      = float(cv_results["mean_test_accuracy"][best_idx])

    best_est = search.best_estimator_
    y_pred = best_est.predict(X_test)

    test_f1       = float(f1_score(y_test, y_pred, average="macro"))
    test_f1_w     = float(f1_score(y_test, y_pred, average="weighted"))
    test_acc  = float(accuracy_score(y_test, y_pred))
    test_bacc     = float(balanced_accuracy_score(y_test, y_pred))
    test_recall_m = float(recall_score(y_test, y_pred, average="macro"))
    test_acc = float(accuracy_score(y_test, y_pred))

    return dict(
        mejores_params=search.best_params_,
        tiempo=float(elapsed),
        n_iter_efectivo=int(n_iter_efectivo),
        n_combinaciones_posibles=(int(n_posibles) if n_posibles is not None else None),
        cv=dict(
            f1=cv_f1,
            f1_w=cv_f1_w,
            bacc=cv_bacc,
            recall_macro=cv_recall_m,
            accuracy=cv_acc,
        ),
        test=dict(
            f1=test_f1,
            f1_w=test_f1_w,
            bacc=test_bacc,
            recall_macro=test_recall_m,
            accuracy=test_acc,
        ),
    )


In [21]:
N_ITER_BUSQUEDA_POR_DEFECTO = 4

# =========================
# Utilidades de datos
# =========================
def cargar_matriz_caracteristicas_y_etiquetas_desde_csv(ruta_csv):
    """Lee un CSV y devuelve (X, y). Usa 'target' si existe, si no la última columna como y."""
    df = pd.read_csv(ruta_csv)
    if "target" in df.columns:
        X = df.drop(columns=["target"]).to_numpy(dtype=np.float32, copy=False)
        y = df["target"].to_numpy()
    else:
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32, copy=False)
        y = df.iloc[:, -1].to_numpy()
    return X, y


def definir_configuracion_busqueda_para_dataset(X_train, nombre_dataset_logico, tipo_combination):
    """
    Define configuración de búsqueda SOLO para el tuning de base.
    - Shuttle aumentado -> CV=2 (no aplica si solo tuneamos base)
    - Shuttle o n>=10000 -> CV=3
    - resto -> CV=5
    """
    n_muestras = X_train.shape[0]
    es_shuttle = nombre_dataset_logico.lower() == "shuttle"

    # Como tuneamos SOLO base, este branch normalmente no se usa.
    if es_shuttle and tipo_combination != "base":
        cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
    elif es_shuttle or n_muestras >= 10000:
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    else:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    return {
        "cv": cv,
        "n_iter": N_ITER_BUSQUEDA_POR_DEFECTO,
        "n_jobs": 1,
    }


def entrenar_y_evaluar_con_params_fijos(
    estimator,
    mejores_params,
    X_train,
    y_train,
    X_test,
    y_test,
):
    """
    Entrena un estimador con hiperparámetros fijos y evalúa en Test.
    Retorna métricas (solo Test) y el tiempo de entrenamiento+predicción.
    """
    inicio = time.perf_counter()

    estimator.set_params(**mejores_params)
    estimator.fit(X_train, y_train)

    y_pred = estimator.predict(X_test)

    elapsed = time.perf_counter() - inicio

    test_f1 = float(f1_score(y_test, y_pred, average="macro"))
    test_f1_w = float(f1_score(y_test, y_pred, average="weighted"))
    test_bacc = float(balanced_accuracy_score(y_test, y_pred))
    test_recall_m = float(recall_score(y_test, y_pred, average="macro"))
    test_acc = float(accuracy_score(y_test, y_pred))

    return dict(
        tiempo=float(elapsed),
        test=dict(
            f1=test_f1,
            f1_w=test_f1_w,
            accuracy=test_acc,
            bacc=test_bacc,
            recall_macro=test_recall_m,
        ),
    )


def _clave_contexto(combo, nombre_modelo):
    # clave por dataset + modelo + grado de limpieza (I)
    return (combo.dataset_logico.lower(), nombre_modelo, str(combo.grado_limpieza))


# =========================
# Paso 1: TUNING SOLO BASE
# =========================
best_params_por_contexto = {}
ruta_best_params = os.path.join(DIRECTORIO_SALIDA, "best_params_por_contexto.json")
total_base = len(set(_clave_contexto(combo, nombre_modelo)
                     for nombre_modelo, combo in tareas
                     if combo.tipo_combination == "base"))

contador_base = 0

print("\n" + "="*100)
print("🎯 ETAPA 1: TUNING SOLO EN 'BASE' (por dataset + modelo + I)")
print("="*100)

for nombre_modelo, combo in tareas:

    if combo.tipo_combination != "base":
        continue

    clave = _clave_contexto(combo, nombre_modelo)

    if clave in best_params_por_contexto:
        continue

    contador_base += 1

    print(f"\n{'='*80}")
    print(f"🎯 TUNING BASE [{contador_base}/{total_base}] Dataset: {combo.dataset_logico} | "
          f"Modelo: {nombre_modelo} | I={combo.grado_limpieza}")
    print(f"📂 Train(base): {os.path.basename(combo.ruta_train_csv)}")
    print(f"📂 Test (base): {os.path.basename(combo.ruta_test_csv)}")

    # cargar base
    try:
        X_train, y_train = cargar_matriz_caracteristicas_y_etiquetas_desde_csv(combo.ruta_train_csv)
        X_test,  y_test  = cargar_matriz_caracteristicas_y_etiquetas_desde_csv(combo.ruta_test_csv)
    except Exception as e:
        print(f"❌ Error leyendo CSV base: {e}")
        continue

    configuracion_busqueda = definir_configuracion_busqueda_para_dataset(
        X_train, combo.dataset_logico, combo.tipo_combination
    )

    estimator, space = REGISTRO_MODELOS[nombre_modelo]()

    print(f"⚙️  RS(base): n_iter={configuracion_busqueda['n_iter']} (auto-ajustado dentro de ejecutar_rs), "
          f"folds={configuracion_busqueda['cv'].n_splits}, n_jobs={configuracion_busqueda['n_jobs']}")
    print("🚀 Iniciando RandomizedSearchCV en BASE...")

    try:
        resultados = ejecutar_rs_y_comparar_cv_con_test(
            estimator, space, X_train, y_train, X_test, y_test,
            configuracion_busqueda=configuracion_busqueda,
            verbose=1
        )
    except Exception as e:
        print(f"❌ Error durante RS(base): {e}")
        continue

    best_params_por_contexto[clave] = {
        "mejores_params": resultados["mejores_params"],
        "cv_f1_macro": resultados["cv"]["f1"],
        "cv_f1_weighted": resultados["cv"]["f1_w"],
        "cv_accuracy": resultados["cv"]["accuracy"],
        "cv_balanced_accuracy": resultados["cv"]["bacc"],
        "cv_recall_macro": resultados["cv"]["recall_macro"],
    }


    # guardado incremental por si se corta
    try:
        serializable = {str(k): v for k, v in best_params_por_contexto.items()}
        with open(ruta_best_params, "w", encoding="utf-8") as f:
            json.dump(serializable, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"⚠️ No se pudo guardar best_params JSON: {e}")

    print(f"✅ RS(base) completado en {resultados['tiempo']:.2f} s | n_iter efectivo={resultados.get('n_iter_efectivo', '--')}")
    print(f"📊 BASE F1(CV): {resultados['cv']['f1']:.4f} | F1(Test): {resultados['test']['f1']:.4f}")
    print(f"🧠 Best params: {resultados['mejores_params']}")


print(f"DEBUG: tunings ejecutados = {len(best_params_por_contexto)} | esperado = {total_base}")


if not best_params_por_contexto:
    raise RuntimeError("❌ No se obtuvo ningún best_params (no se encontró base o falló el tuning).")


# =========================
# Paso 2: EVALUACIÓN MASIVA con params fijos
# =========================
registros = []
inicio_total = time.perf_counter()

print("\n" + "="*100)
print("🏁 ETAPA 2: EVALUACIÓN MASIVA (params fijos) sobre TODOS los train CSV generados")
print("="*100)

for idx, (nombre_modelo, combo) in enumerate(tareas, start=1):

    print(f"\n{'='*80}")
    print(f"🏁 [{idx}/{total_tareas}] Dataset: {combo.dataset_logico} | "
          f"Tipo: {combo.tipo_combination} | Modelo: {nombre_modelo}")
    print(f"📂 Train: {os.path.basename(combo.ruta_train_csv)}")

    clave = _clave_contexto(combo, nombre_modelo)
    if clave not in best_params_por_contexto:
        print(f"⚠️ Sin best_params para {clave}. Se omite.")
        continue

    contexto = best_params_por_contexto[clave]    
    mejores_params = contexto["mejores_params"]

    # cargar train/test (de cada combinación)
    try:
        X_train, y_train = cargar_matriz_caracteristicas_y_etiquetas_desde_csv(combo.ruta_train_csv)
        X_test,  y_test  = cargar_matriz_caracteristicas_y_etiquetas_desde_csv(combo.ruta_test_csv)
    except Exception as e:
        print(f"❌ Error leyendo CSV: {e}")
        continue

    estimator, _space = REGISTRO_MODELOS[nombre_modelo]()

    # entrenar + evaluar (sin RS)
    try:
        resultados = entrenar_y_evaluar_con_params_fijos(
            estimator=estimator,
            mejores_params=mejores_params,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
        )
    except Exception as e:
        print(f"❌ Error durante fit/eval params fijos: {e}")
        continue

    print(f"✅ Eval completada en {resultados['tiempo']:.2f} s")
    print(f"📊 F1(Test): {resultados['test']['f1']:.4f}")

    # registrar resultados (CV ya no aplica por combinación: NaN)
    registros.append(asdict(RegistroRendimiento(
        dataset_logico=combo.dataset_logico,
        tipo_combination=combo.tipo_combination,
        nombre_modelo_aprendizaje=nombre_modelo,
        nombre_configuracion=combo.nombre_configuracion,
        tecnica_aumento=combo.tecnica_aumento,
        valor_densidad=str(combo.valor_densidad),
        valor_riesgo=str(combo.valor_riesgo),
        criterio_pureza=str(combo.criterio_pureza),
        grado_limpieza=str(combo.grado_limpieza),

        cantidad_train=int(X_train.shape[0]),
        cantidad_test=int(X_test.shape[0]),
        cantidad_caracteristicas=int(X_train.shape[1]),

        cv_f1_macro=round(contexto["cv_f1_macro"], 3),
        cv_f1_weighted=round(contexto["cv_f1_weighted"], 3),
        cv_balanced_accuracy=float("nan"),
        cv_recall_macro=float("nan"),
        cv_accuracy=round(contexto.get("cv_accuracy", 0), 3),

        test_f1_macro=round(resultados["test"]["f1"], 3),
        test_f1_weighted=round(resultados["test"]["f1_w"], 3),
        test_balanced_accuracy=round(resultados["test"]["bacc"], 3),
        test_recall_macro=round(resultados["test"]["recall_macro"], 3),
        test_accuracy=round(resultados["test"]["accuracy"], 3),
        
        mejores_hiperparametros=str(mejores_params),
        tiempo_busqueda_seg=float(resultados["tiempo"]),  # ahora es tiempo fit+pred
    )))

    gc.collect()

fin_total = time.perf_counter() - inicio_total
print(f"\n⏱️ Tiempo total etapa train/test (params fijos): {fin_total/60:.2f} min")



🎯 ETAPA 1: TUNING SOLO EN 'BASE' (por dataset + modelo + I)

🎯 TUNING BASE [1/7] Dataset: shuttle | Modelo: RandomForest | I=0
📂 Train(base): shuttle_I0_tm46400_train.csv
📂 Test (base): shuttle_tdataset58000_tm11600_test.csv
⚙️  RS(base): n_iter=4 (auto-ajustado dentro de ejecutar_rs), folds=3, n_jobs=1
🚀 Iniciando RandomizedSearchCV en BASE...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
✅ RS(base) completado en 163.44 s | n_iter efectivo=4
📊 BASE F1(CV): 0.9626 | F1(Test): 0.8549
🧠 Best params: {'classifier__n_estimators': 300, 'classifier__max_features': 'sqrt'}

🎯 TUNING BASE [2/7] Dataset: shuttle | Modelo: RandomForest | I=1
📂 Train(base): shuttle_I1_tm45932_train.csv
📂 Test (base): shuttle_tdataset58000_tm11600_test.csv
⚙️  RS(base): n_iter=4 (auto-ajustado dentro de ejecutar_rs), folds=3, n_jobs=1
🚀 Iniciando RandomizedSearchCV en BASE...
Fitting 3 folds for each of 4 candidates, totalling 12 fits
✅ RS(base) completado en 153.44 s | n_iter efectivo=4
📊 BASE F1(C

In [25]:
# =========================
# EXPORT FINAL A EXCEL (Con seguimiento de PCSMOTE)
# =========================
inicio_total = time.perf_counter()

# Lista de datasets de interés (ajustada a tu pedido)
datasets_interes = ["shuttle", "telco_churn", "us_crime"]

df_resultados = pd.DataFrame(registros).copy()
df_resultados.reset_index(drop=True, inplace=True)

# Normalizar tipos numéricos
columnas_metricas = ["test_f1_macro", "test_f1_weighted", "test_accuracy", 
                     "cv_f1_macro", "cv_f1_weighted", "cv_accuracy"]
for col in columnas_metricas:
    if col in df_resultados.columns:
        df_resultados[col] = pd.to_numeric(df_resultados[col], errors="coerce")

def _tomar_una_fila(df_filtrado):
    if df_filtrado is None or len(df_filtrado) == 0:
        return None
    return df_filtrado.iloc[-1]

resultados_formateados = []
dbg_total = 0
dbg_ok = 0
dbg_empty = 0
dbg_ignorados = 0
dbg_pcsmote_ok = 0 # Contador específico para PCSMOTE

print(f"\n--- [INICIO DE PROCESAMIENTO] ---")

for comb in combinaciones:
    if comb.dataset_logico not in datasets_interes:
        dbg_ignorados += 1
        continue

    dbg_total += 1
    df_fila = pd.DataFrame()
    grado_str = str(comb.grado_limpieza).strip()

    # --- LÓGICA DE MATCH ---
    if comb.tipo_combination == "pcsmote":
        df_fila = df_resultados[
            (df_resultados["tipo_combination"] == "pcsmote") &
            (df_resultados["nombre_configuracion"] == comb.nombre_configuracion)
        ]
    elif comb.tipo_combination == "clasico":
        df_fila = df_resultados[
            (df_resultados["tipo_combination"] == "clasico") &
            (df_resultados["dataset_logico"] == comb.dataset_logico) &
            (df_resultados["grado_limpieza"].astype(str).str.strip() == grado_str) &
            (df_resultados["tecnica_aumento"] == comb.tecnica_aumento)
        ]
    else: # base
        df_fila = df_resultados[
            (df_resultados["tipo_combination"] == "base") &
            (df_resultados["dataset_logico"] == comb.dataset_logico) &
            (df_resultados["grado_limpieza"].astype(str).str.strip() == grado_str)
        ]

    fila = _tomar_una_fila(df_fila)

    if fila is None:
        dbg_empty += 1
        print(f"[ALERTA MATCH] Sin datos para: {comb.dataset_logico} | Tipo: {comb.tipo_combination} | I: {grado_str}")
        continue

    # --- MATCH EXITOSO ---
    dbg_ok += 1
    
    # NUEVO: Debug específico para PCSMOTE
    if comb.tipo_combination == "pcsmote":
        dbg_pcsmote_ok += 1
        print(f"✨ [MATCH PCSMOTE OK] Exp: {comb.nombre_configuracion} | DS: {comb.dataset_logico} | F1-Test: {fila.get('test_f1_macro', 0):.4f}")

    tecnica = str(fila.get("tecnica_aumento", "base"))
    
    # Manejo de semillas y sintéticos con getattr seguro
    sc_analizadas = getattr(comb, "semillas_analizadas", np.nan) if comb.tipo_combination == "pcsmote" else np.nan
    sv_validas = getattr(comb, "semillas_validas", np.nan) if comb.tipo_combination == "pcsmote" else np.nan
    sinteticos_generados = getattr(comb, "sinteticos_generados", np.nan) if comb.tipo_combination != "base" else np.nan

    resultados_formateados.append({
        "tecnica": tecnica,
        "dataset": comb.dataset_logico,
        "percentil_radio_distancia": getattr(comb, "percentil_radio_distancia", np.nan),
        "percentil_riesgo": getattr(comb, "percentil_riesgo", np.nan),
        "criterio_pureza": getattr(comb, "criterio_pureza", np.nan),
        "umbral_densidad": getattr(comb, "umbral_densidad", np.nan),
        "umbral_riesgo": getattr(comb, "umbral_riesgo", np.nan),
        "tipo_pureza": getattr(comb, "tipo_pureza", np.nan),
        "grado_isolation_forest": grado_str,
        "train_n": int(fila.get("cantidad_train", 0)),
        "semillas_sinteticas_candidatas": sc_analizadas,
        "semillas_candidatas_validas_train": sv_validas,
        "sinteticas_generadas": sinteticos_generados,
        "f1_macro_cv" : float(fila.get("cv_f1_macro", np.nan)), 
        "f1_weighted_cv" : float(fila.get("cv_f1_weighted", np.nan)), 
        "accuracy_cv" : float(fila.get("cv_accuracy", np.nan)),
        "f1_macro_test": float(fila.get("test_f1_macro", np.nan)),
        "f1_weighted_test": float(fila.get("test_f1_weighted", np.nan)),
        "accuracy_test": float(fila.get("test_accuracy", np.nan)),
    })

print(f"\n--- [RESUMEN FINAL] ---")
print(f"Ignorados (otros datasets): {dbg_ignorados}")
print(f"Total analizados (interés): {dbg_total}")
print(f"  - Exitosos totales: {dbg_ok}")
print(f"  - Exitosos PCSMOTE: {dbg_pcsmote_ok} ✅")
print(f"  - Fallos en interés: {dbg_empty}")

# Ejecuta esto para ver qué hay realmente en el campo 'tipo_combination'
print("Valores únicos en 'tipo_combination':", df_resultados['tipo_combination'].unique())

# Si PCSMOTE está dentro de 'clasico' por error, lo buscamos por nombre de técnica
if 'tecnica_aumento' in df_resultados.columns:
    print("Técnicas encontradas:", df_resultados['tecnica_aumento'].unique())

# Buscar si existe la palabra 'pcsmote' en cualquier parte del DataFrame
mask_pcsmote = df_resultados.stack().str.contains('pcsmote', case=False, na=False).unstack().any(axis=1)
print(f"¿Hay alguna fila que mencione 'pcsmote' en algún lado?: {mask_pcsmote.any()}")

if resultados_formateados:
    df_export = pd.DataFrame(resultados_formateados)
    df_export.to_excel(NOMBRE_ARCHIVO_EXCEL, index=False)
    print(f"✅ Archivo generado: {NOMBRE_ARCHIVO_EXCEL}")


--- [INICIO DE PROCESAMIENTO] ---
✨ [MATCH PCSMOTE OK] Exp: PRD70_PR30_CPprop_UD025_UR030_Upp040_I1_SC9828_SV7685_SG206796 | DS: shuttle | F1-Test: 0.9500
✨ [MATCH PCSMOTE OK] Exp: PRD70_PR30_CPprop_UD035_UR030_Upp045_I1_SC9828_SV7105_SG206796 | DS: shuttle | F1-Test: 0.9500
✨ [MATCH PCSMOTE OK] Exp: PRD70_PR30_CPprop_UD040_UR035_Upp050_I1_SC9828_SV7105_SG206796 | DS: shuttle | F1-Test: 0.9500
✨ [MATCH PCSMOTE OK] Exp: PRD70_PR30_CPprop_UD050_UR040_Upp060_I0_SC9931_SV6756_SG135963 | DS: shuttle | F1-Test: 0.8570
✨ [MATCH PCSMOTE OK] Exp: PRD70_PR30_CPprop_UD050_UR040_Upp060_I1_SC1480_SV453_SG2617 | DS: telco_churn | F1-Test: 0.7110
✨ [MATCH PCSMOTE OK] Exp: PRD85_PR30_CPprop_UD060_UR040_Upp060_I1_SC1480_SV495_SG2617 | DS: telco_churn | F1-Test: 0.7150
✨ [MATCH PCSMOTE OK] Exp: PRD85_PR40_CPent_UD060_UR045_PE70_I0_SC1495_SV897_SG2644 | DS: telco_churn | F1-Test: 0.7080
✨ [MATCH PCSMOTE OK] Exp: PRD85_PR50_CPprop_UD050_UR055_Upp045_I1_SC1480_SV760_SG2617 | DS: telco_churn | F1-Test: 0.7